In [ ]:
!pip install git+https://github.com/huggingface/transformers.git sentencepiece


# Meta SeamlessM4T

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2Model
import torchaudio
import torch

# Load model + processor
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large")

# Load audio
audio, orig_freq = torchaudio.load("sample.wav")

# Resample to 16kHz (required)
audio = torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16000)

# Convert to mono if needed
if audio.shape[0] > 1:
    audio = audio.mean(dim=0)



In [ ]:
# Preprocess
inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")

# 🔥 IMPORTANT: force TEXT output
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        tgt_lang="hin",          # language of transcription
        generate_speech=False   # <-- THIS enables ASR
    )

# Decode text
transcription = processor.decode(generated_tokens[0], skip_special_tokens=True)

print(transcription)

['दया और मालूमियत की भारत कण-उत्पादनी की ताकत है, एक नन्दी-हरी रीति है, अंत में भारत के लोगों की आय कहा जाता है, उन लोगों के कानून को जानने वाली आई है।']


In [ ]:
!pip install datasets torchaudio jiwer

In [ ]:
from datasets import load_from_disk
from transformers import AutoProcessor, SeamlessM4Tv2Model
import torch
import torchaudio
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, Strip
from tqdm import tqdm

# Load dataset
dataset = load_from_disk(<test_dat>)

# Load model
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()



In [ ]:
len(dataset)

500

In [ ]:
dataset["sentence"][20]

'इनिः लोः दो आञ सोबेन सोमाए कमि अङगव जञ तइकेनाञ।'

In [ ]:

predictions = []
references = []

target_language_code = "hin" # change here


for sample in tqdm(dataset):
    # --- Audio ---
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Convert to tensor
    audio = torch.tensor(audio)

    # Resample if needed
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)

    # --- Preprocess ---
    inputs = processor(
        audio=audio.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    # --- ASR using SeamlessM4T ---
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            tgt_lang=target_language_code,          # change if needed
            generate_speech=False
        )

    pred_text = processor.decode(output_tokens[0], skip_special_tokens=True)

    # --- Reference ---
    ref_text = sample["sentence"]

    predictions.append(pred_text)
    references.append(ref_text)

references = [r[0] if isinstance(r, list) else r for r in references]
predictions = [p[0] if isinstance(p, list) else p for p in predictions]
error = wer(references, predictions)

In [ ]:
predictions

In [ ]:
references = [r[0] if isinstance(r, list) else r for r in references]
predictions = [p[0] if isinstance(p, list) else p for p in predictions]
error = wer(references, predictions)

In [ ]:
print(f"WER: {error}")

WER: 0.9477254588986432


Meta MMS 1B

In [ ]:
import torch
import torchaudio
from tqdm import tqdm
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip
from transformers import Wav2Vec2ForCTC, AutoProcessor

# --- Load model ---
model_id = "facebook/mms-1b-all"
processor = AutoProcessor.from_pretrained(model_id)
model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
model.eval()


In [ ]:

# IMPORTANT: set language (very important for MMS)
processor.tokenizer.set_target_lang("hin")   # change if needed

# --- Normalization ---
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

predictions = []
references = []

for sample in tqdm(dataset):
    # --- Audio ---
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    audio = torch.tensor(audio)

    # Resample
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)

    # --- Preprocess ---
    inputs = processor(
        audio.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).to(device)

    # --- Forward pass (CTC) ---
    with torch.no_grad():
        logits = model(**inputs).logits

    # --- Decode ---
    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    # --- Reference ---
    ref_text = sample["sentence"]

    predictions.append(transform(pred_text))
    references.append(transform(ref_text))



100%|██████████| 500/500 [02:06<00:00,  3.95it/s]


In [ ]:
predictions

In [ ]:
# --- WER ---
error = wer(references, predictions)
print(f"\nWER: {error:.4f}")


WER: 1.0625


In [ ]:
del model
del processor
#del pipe  # if you used pipeline

Whisper large v3

In [ ]:
import torch
import torchaudio
from tqdm import tqdm
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

# --- Device ---
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# --- Load model ---
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
).to(device)

processor = AutoProcessor.from_pretrained(model_id)


In [ ]:
# Load dataset
from datasets import load_from_disk
dataset = load_from_disk("<test_data>")

In [ ]:

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

# --- Normalization ---
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

predictions = []
references = []

for sample in tqdm(dataset):
    # --- Audio ---
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    # Convert to tensor
    audio = torch.tensor(audio)

    # Resample to 16k (Whisper expects this)
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)

    # Convert back to numpy (pipeline expects numpy)
    audio_input = {
        "array": audio.numpy(),
        "sampling_rate": 16000
    }

    # --- Inference ---
    result = pipe(
        audio_input,
        generate_kwargs={"language": "hindi"}  # change if needed
    )

    pred_text = result["text"]

    # --- Reference ---
    ref_text = sample["sentence"]  # or "text" depending on dataset

    predictions.append(transform(pred_text))
    references.append(transform(ref_text))

# --- WER ---
error = wer(references, predictions)
print(f"\nWER: {error:.4f}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
  0%|          | 0/500 [00:00<?, ?it/s][transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformer


WER: 0.8962


Wav2vec2 xlsr 1B

In [ ]:
import torch
import torchaudio
from tqdm import tqdm
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip
from transformers import Wav2Vec2ForCTC, AutoProcessor

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load XLS-R with CTC ---
model_id = "facebook/wav2vec2-xls-r-1b"

processor = AutoProcessor.from_pretrained(model_id)
model = Wav2Vec2ForCTC.from_pretrained(model_id).to(device)
model.eval()


In [ ]:

# --- Normalization ---
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

predictions = []
references = []

for sample in tqdm(dataset):
    # --- Audio ---
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    audio = torch.tensor(audio)

    # Resample to 16k
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)

    # --- Preprocess ---
    inputs = processor(
        audio.numpy(),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    ).to(device)

    # --- Forward ---
    with torch.no_grad():
        logits = model(**inputs).logits

    # --- Decode (CTC) ---
    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    # --- Reference ---
    ref_text = sample["sentence"]  # adjust if needed

    predictions.append(transform(pred_text))
    references.append(transform(ref_text))

# --- WER ---
error = wer(references, predictions)
print(f"\nWER: {error:.4f}")

SpeechT5

In [ ]:
import torch
import torchaudio
from tqdm import tqdm
from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load model ---
model_id = "microsoft/speecht5_asr"

processor = SpeechT5Processor.from_pretrained(model_id)
model = SpeechT5ForSpeechToText.from_pretrained(model_id).to(device)
model.eval()

# --- Normalization ---
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

predictions = []
references = []

for sample in tqdm(dataset):
    # --- Audio ---
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    audio = torch.tensor(audio)

    # Resample to 16k if needed
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)


      # --- Preprocess ---
    inputs = processor(
        audio=audio.numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    # --- Inference ---
    with torch.no_grad():
        predicted_ids = model.generate(**inputs, max_length=100)

    pred_text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]





    # --- Reference ---
    ref_text = sample["sentence"]  # change to "text" if needed

    predictions.append(transform(pred_text))
    references.append(transform(ref_text))

# --- WER ---
error = wer(references, predictions)
print(f"\nWER: {error:.4f}")